# Sqrt Decomposition & Offline Queries

The **middle ground** for range queries. **Prefix sums** (the prefix-sums notebook) give O(1) queries but O(n) updates; **Fenwick & segment trees** give O(log n) for both. **Sqrt decomposition** splits the array into ~√n blocks and gets **O(√n)** for both — bigger than O(log n), but the code is dead simple and the constant is tiny, so it often *wins in practice* and bends to operations a segment tree can't express. Its offline cousin, **Mo's algorithm**, reorders a batch of queries so a sliding window answers all of them in O((n+q)√n) — the payoff of *not* answering queries in the order they arrive.

**Contents**

1. **The √n idea** — why block size √n balances update and query cost
2. **Sqrt decomposition** — range-sum + point update in O(√n), from scratch
3. **Block lazy** — O(√n) *range* updates too
4. **Mo's algorithm** — offline range queries by reordering, from scratch
5. **Offline processing** — the general "sort then sweep" idea
6. **When to use** (sqrt vs segment tree vs Mo's)
7. **Complexity cheat-sheet**

## 1. The √n idea

Sqrt decomposition cuts the array into contiguous **blocks of size ≈ √n** and keeps one summary per block (here, the block's sum). The two costs then pull in opposite directions:

- A **point update** touches one element and refreshes its block's summary — **O(1)**.
- A **range query** over `[l, r]` walks the at-most-two *partial* blocks at the ends element-by-element, and jumps over every *whole* block in the middle using its summary.

If a block holds `b` elements there are `n/b` blocks, so a query costs `O(b + n/b)`. That sum is minimized when the two terms are equal — `b = n/b`, i.e. **`b = √n`** — giving **O(√n)** per query. That's the whole trick, and it sits squarely between the two extremes:

| Approach | Range query | Point update |
|---|:---:|:---:|
| **prefix sums** (static) | O(1) | O(n) rebuild |
| **sqrt decomposition** | O(√n) | O(1)–O(√n) |
| **fenwick / segment tree** | O(log n) | O(log n) |

O(√n) is asymptotically worse than O(log n), but the loops are plain array scans with no recursion or pointer chasing, so the hidden constant is small — and, unlike a Fenwick tree, the block scan can compute things sums can't (a range's distinct count, median, mode).

In [1]:
import math

# Why b = sqrt(n): the cost model b + n/b is minimized at b = sqrt(n).
n = 10_000
for b in (1, 10, 50, 100, 200, 1000, n):
    cost = b + n // b                      # ~partial-block scan + ~whole-block jumps
    print(f"block size {b:>6}: ~{cost:>6} ops/query")

best = round(math.isqrt(n))
print(f"\nsqrt(n) = {best}  ->  ~{best + n // best} ops/query (the minimum)")

block size      1: ~ 10001 ops/query
block size     10: ~  1010 ops/query
block size     50: ~   250 ops/query
block size    100: ~   200 ops/query
block size    200: ~   250 ops/query
block size   1000: ~  1010 ops/query
block size  10000: ~ 10001 ops/query

sqrt(n) = 100  ->  ~200 ops/query (the minimum)


## 2. Sqrt decomposition — range sum + point update

The classic build: an array `a`, plus a `blocks[]` array holding the running sum of each √n-sized block. A point update fixes one cell *and* its block sum in O(1). A range query adds up the two partial end-blocks element-wise and the whole middle blocks via `blocks[]` — O(√n).

In [2]:
import math

class SqrtDecomp:
    def __init__(self, data):
        self.a = list(data)
        self.n = len(self.a)
        self.bs = max(1, math.isqrt(self.n))            # block size ~ sqrt(n)
        self.nb = (self.n + self.bs - 1) // self.bs     # number of blocks
        self.blocks = [0] * self.nb
        for i, x in enumerate(self.a):                  # O(n) build
            self.blocks[i // self.bs] += x

    def update(self, i, value):                         # point assign, O(1)
        self.blocks[i // self.bs] += value - self.a[i]
        self.a[i] = value

    def range_sum(self, l, r):                          # inclusive [l, r], O(sqrt n)
        bl, br = l // self.bs, r // self.bs
        if bl == br:                                    # same block: just scan it
            return sum(self.a[l:r + 1])
        s = sum(self.a[l:(bl + 1) * self.bs])           # left partial block
        for b in range(bl + 1, br):                     # whole interior blocks
            s += self.blocks[b]
        s += sum(self.a[br * self.bs:r + 1])            # right partial block
        return s

sd = SqrtDecomp([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print("range_sum(2, 7) :", sd.range_sum(2, 7))         # 3+4+5+6+7+8 = 33
sd.update(4, 50)                                        # a[4] = 50, O(1)
print("after a[4]=50, range_sum(0, 9):", sd.range_sum(0, 9))   # 55 - 5 + 50 = 100

range_sum(2, 7) : 33
after a[4]=50, range_sum(0, 9): 100


**Proof vs brute force.** Interleave random point-updates and random range-sum queries, checking every answer against a plain Python list. Seeded, so the run is deterministic.

In [3]:
import random
random.seed(42)

ref = [random.randint(-20, 20) for _ in range(97)]      # odd length -> ragged last block
sd  = SqrtDecomp(ref[:])
for _ in range(3000):
    if random.random() < 0.5:                           # point update
        i, v = random.randrange(len(ref)), random.randint(-20, 20)
        ref[i] = v
        sd.update(i, v)
    else:                                               # range-sum query
        l = random.randrange(len(ref)); r = random.randint(l, len(ref) - 1)
        assert sd.range_sum(l, r) == sum(ref[l:r + 1])
print("3000 interleaved update / range-sum ops vs brute force: OK")

3000 interleaved update / range-sum ops vs brute force: OK


## 3. Block lazy — O(√n) range updates too

To also support **range updates** ("add `v` to every element in `[l, r]`") in O(√n), give each block a **lazy add** that applies to all its elements at once. A range-add touches the two partial end-blocks element-wise (and refreshes their block sums) and bumps each whole interior block's `lazy` in O(1) — same √n shape as a query. The mechanism mirrors the lazy propagation in the **fenwick & segment trees** notebook, but flat and far simpler.

In [4]:
import math

class SqrtDecompLazy:
    def __init__(self, data):
        self.a = list(data)
        self.n = len(self.a)
        self.bs = max(1, math.isqrt(self.n))
        self.nb = (self.n + self.bs - 1) // self.bs
        self.bsum = [0] * self.nb           # sum of (raw) elements in the block
        self.lazy = [0] * self.nb           # pending add applied to every element
        for i, x in enumerate(self.a):
            self.bsum[i // self.bs] += x

    def range_add(self, l, r, v):           # add v to [l, r], O(sqrt n)
        bl, br = l // self.bs, r // self.bs
        if bl == br:
            for i in range(l, r + 1):
                self.a[i] += v; self.bsum[bl] += v
            return
        for i in range(l, (bl + 1) * self.bs):          # left partial: touch elements
            self.a[i] += v; self.bsum[bl] += v
        for b in range(bl + 1, br):                      # whole blocks: just the lazy tag
            self.lazy[b] += v
        for i in range(br * self.bs, r + 1):            # right partial
            self.a[i] += v; self.bsum[br] += v

    def range_sum(self, l, r):              # inclusive [l, r], O(sqrt n)
        bl, br = l // self.bs, r // self.bs
        if bl == br:
            return sum(self.a[i] + self.lazy[bl] for i in range(l, r + 1))
        s = sum(self.a[i] + self.lazy[bl] for i in range(l, (bl + 1) * self.bs))
        for b in range(bl + 1, br):
            s += self.bsum[b] + self.lazy[b] * self.bs  # block sum + lazy * block width
        s += sum(self.a[i] + self.lazy[br] for i in range(br * self.bs, r + 1))
        return s

import random
random.seed(7)
ref = [0] * 80
sd  = SqrtDecompLazy(ref[:])
for _ in range(2000):
    l = random.randrange(80); r = random.randint(l, 79); v = random.randint(-9, 9)
    for i in range(l, r + 1): ref[i] += v
    sd.range_add(l, r, v)
    a = random.randrange(80); b = random.randint(a, 79)
    assert sd.range_sum(a, b) == sum(ref[a:b + 1])
print("2000 random range-add / range-sum ops vs brute force: OK")

2000 random range-add / range-sum ops vs brute force: OK


## 4. Mo's algorithm — offline range queries by reordering

Some range queries have **no easy point-update structure** — the textbook one is *"how many **distinct** values are in `a[l..r]`?"*. A Fenwick/segment tree can't combine distinct-counts of two halves (it isn't associative over a simple summary). But if you have **all the queries up front** (they're *offline*), you can maintain a sliding window `[curL, curR]` with O(1) `add(x)` / `remove(x)` on a frequency table, and *move* the window from one query's range to the next.

Naively that re-walks the array per query. Mo's trick is the **ordering**: sort queries by **(block of L, then R)** — with R ascending in even blocks and descending in odd (the "Hilbert-lite" / snake order) to avoid R bouncing. With block size √n the left pointer moves O(√n) per query and the right pointer moves O(n) per block over O(√n) blocks, for a total of **O((n + q)·√n)**.

In [5]:
import math
from collections import defaultdict

def mos_distinct(a, queries):
    # answer each (l, r) inclusive query: count of distinct values in a[l..r]
    n, q = len(a), len(queries)
    bs = max(1, math.isqrt(n))
    # sort query *indices* by (block of l, then r) with snake order on r
    order = sorted(range(q), key=lambda i: (
        queries[i][0] // bs,
        queries[i][1] if (queries[i][0] // bs) % 2 == 0 else -queries[i][1],
    ))
    freq = defaultdict(int)
    distinct = 0
    ans = [0] * q
    curL, curR = 0, -1                      # empty window

    def add(x):
        nonlocal distinct
        if freq[x] == 0: distinct += 1
        freq[x] += 1

    def remove(x):
        nonlocal distinct
        freq[x] -= 1
        if freq[x] == 0: distinct -= 1

    for i in order:
        l, r = queries[i]
        while curR < r: curR += 1; add(a[curR])     # grow right
        while curL > l: curL -= 1; add(a[curL])     # grow left
        while curR > r: remove(a[curR]); curR -= 1  # shrink right
        while curL < l: remove(a[curL]); curL += 1  # shrink left
        ans[i] = distinct
    return ans

a = [1, 1, 2, 1, 3, 2, 1, 4]
qs = [(0, 4), (1, 1), (2, 7), (0, 7), (3, 5)]
print("distinct counts:", mos_distinct(a, qs))
# [0,4]->{1,2,3}=3, [1,1]->{1}=1, [2,7]->{2,1,3,4}=4, [0,7]->{1,2,3,4}=4, [3,5]->{1,3,2}=3

distinct counts: [3, 1, 4, 4, 3]


**Proof vs brute force.** Every Mo's answer must match `len(set(a[l:r+1]))` computed directly, for a random array and a batch of random ranges. Seeded.

In [6]:
import random
random.seed(2024)

a  = [random.randint(0, 9) for _ in range(200)]         # low cardinality -> real dedup
qs = [tuple(sorted((random.randrange(200), random.randrange(200)))) for _ in range(500)]

got = mos_distinct(a, qs)
exp = [len(set(a[l:r + 1])) for (l, r) in qs]
assert got == exp
print("500 distinct-count queries on n=200 vs brute force: OK")

# the same engine answers a different aggregate by swapping add/remove:
# sum of freq^2 ("power" of a range) just tracks the running total.
def mos_freq_sq(a, queries):
    n, q = len(a), len(queries)
    bs = max(1, math.isqrt(n))
    order = sorted(range(q), key=lambda i: (
        queries[i][0] // bs,
        queries[i][1] if (queries[i][0] // bs) % 2 == 0 else -queries[i][1]))
    freq = defaultdict(int); total = 0; ans = [0] * q; curL, curR = 0, -1
    def add(x):
        nonlocal total; total += 2 * freq[x] + 1; freq[x] += 1     # (f+1)^2 - f^2
    def remove(x):
        nonlocal total; freq[x] -= 1; total -= 2 * freq[x] + 1     # f^2 - (f+1)^2
    for i in order:
        l, r = queries[i]
        while curR < r: curR += 1; add(a[curR])
        while curL > l: curL -= 1; add(a[curL])
        while curR > r: remove(a[curR]); curR -= 1
        while curL < l: remove(a[curL]); curL += 1
        ans[i] = total
    return ans

from collections import Counter
got2 = mos_freq_sq(a, qs)
exp2 = [sum(c * c for c in Counter(a[l:r + 1]).values()) for (l, r) in qs]
assert got2 == exp2
print("500 sum-of-freq^2 queries vs brute force: OK")

500 distinct-count queries on n=200 vs brute force: OK
500 sum-of-freq^2 queries vs brute force: OK


## 5. Offline processing — the general idea

Mo's is one instance of a broader strategy: when you're handed **all** the operations in advance, you're free to **answer them in a better order than they arrived**. The recipe:

1. **Collect** every query (tag each with its original index so you can un-shuffle the answers at the end).
2. **Sort / sweep** them along some axis — by left endpoint, by value, by time — so a single pass with a cheap incremental structure handles them all.
3. **Scatter** the answers back into original order.

You've already seen this shape:

- **Sweep line** (the sweep-line notebook) — sort events by x-coordinate, process left to right, keep an active set.
- **Coordinate compression** (the coordinate-compression notebook) — pre-sort the distinct values to remap a huge value range into `0..k`, so an array/Fenwick can index by rank.

Below: a tiny offline sweep for **"how many values in `a[l..r]` are ≤ k?"** Sort queries by `k` and feed elements into a Fenwick (over positions) in increasing value order — each query sees exactly the elements that qualify. (Sorting the queries is the offline move; the per-query cost drops to O(log n).)

In [7]:
# offline: count elements <= k within positions [l, r], answered out of order.
class Fenwick:                              # from the fenwick & segment trees notebook
    def __init__(self, n):
        self.n = n; self.t = [0] * (n + 1)
    def add(self, i, d):
        i += 1
        while i <= self.n: self.t[i] += d; i += i & -i
    def prefix(self, i):                    # sum of [0, i)
        s = 0
        while i > 0: s += self.t[i]; i -= i & -i
        return s
    def range_count(self, l, r):            # count inserted positions in [l, r]
        return self.prefix(r + 1) - self.prefix(l)

def count_le_offline(a, queries):
    n = len(a)
    # process elements from smallest value up; answer a query once k >= all needed values
    elems = sorted(range(n), key=lambda i: a[i])            # positions by ascending value
    order = sorted(range(len(queries)), key=lambda i: queries[i][2])   # queries by k
    fen = Fenwick(n)
    ans = [0] * len(queries)
    e = 0
    for qi in order:
        l, r, k = queries[qi]
        while e < n and a[elems[e]] <= k:   # insert every element with value <= k
            fen.add(elems[e], 1); e += 1
        ans[qi] = fen.range_count(l, r)     # how many of them fall in [l, r]
    return ans

import random
random.seed(11)
a  = [random.randint(0, 50) for _ in range(120)]
qs = [(*sorted((random.randrange(120), random.randrange(120))), random.randint(0, 50))
      for _ in range(300)]
got = count_le_offline(a, qs)
exp = [sum(1 for x in a[l:r + 1] if x <= k) for (l, r, k) in qs]
assert got == exp
print("300 offline 'count <= k in [l,r]' queries vs brute force: OK")

300 offline 'count <= k in [l,r]' queries vs brute force: OK


## 6. When to use what

| You need… | Reach for | Why |
|---|---|---|
| static array, many range-sum queries | **prefix sums** | O(1), nothing fancier needed |
| point update + range **sum** | fenwick tree | O(log n), smallest constant |
| point update + range **min/max/gcd** | segment tree | O(log n), any associative op |
| a range aggregate that **isn't easily associative** (distinct count, mode, k-th) | **sqrt decomposition** | the block scan can compute it directly |
| **all queries known up front**, no updates, weird aggregate | **Mo's algorithm** | O((n+q)√n) by reordering |
| online range work where O(√n) per op is fine and code time matters | **sqrt decomposition** | far simpler to write/debug than a segment tree |

**Rules of thumb.** Reach for a Fenwick/segment tree first when the operation is associative and you need O(log n). Drop to **sqrt decomposition** when the aggregate doesn't fit a tree, or when you want something you can write correctly in five minutes. Switch to **Mo's** only when the queries are *offline* and there's no update between them — its whole speedup comes from being allowed to answer out of order.

## 7. Complexity cheat-sheet

| Technique | Build | Range query | Point update | Range update |
|---|:---:|:---:|:---:|:---:|
| Prefix sums | O(n) | O(1) | O(n) rebuild | — |
| **Sqrt decomposition** | O(n) | **O(√n)** | O(1) | O(√n) *(block lazy)* |
| Fenwick tree | O(n) | O(log n) | O(log n) | — |
| Segment tree (+ lazy) | O(n) | O(log n) | O(log n) | O(log n) |
| **Mo's algorithm** | O(q log q) sort | **O((n+q)√n)** total, offline | — | — |

<sub>Sqrt decomposition is the simple, flexible middle: worse than O(log n) but with a tiny constant and the freedom to compute aggregates a tree can't. Mo's reuses the √n block idea *on the queries* — pay O(q log q) to sort, then slide one window across all of them.</sub>